# Industry Factors — Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for `industries_build.ipynb`, which builds the industry factor exposure layer used by the cross-sectional regression (CSR). USE4 specifies 60 GICS-based industry factors; public data forces material engineering — documented exhaustively in §9 — yielding a factor count somewhat less than 60 (see §5 for the design criteria that determine the right number for your universe).

**The central deliverable is not just parquet files — it is your scheme.** Stage 1 is design work: you will inventory the ~150 classification atoms Sharadar provides, decide how many factors to create and which atoms to group, justify every merge and split against USE4's own published criteria (economic intuition, statistical significance, explanatory power, minimum size), and save the mapping as a version-controlled CSV in this directory. Stages 3–6 are mechanical given the scheme.

**Inputs:** `SHARADAR_TICKERS.csv` (classification), `data/out/estu_monthly.parquet` (from `estu_build.ipynb`), and the scheme CSV you design in Stage 1.

**Deliverables:** `data/out/industries_use4.parquet` (one industry-factor label per stock per date) and `data/out/industry_weights_use4.parquet` (per-date industry cap-weight vector for the CSR's cap-weighted zero-sum constraint).

**Companion specs:** `01_estu/estu_spec.ipynb` (upstream); `04_country_factor/country_spec.ipynb` (downstream, consumes the weights artifact).

## §1. The USE4 industry spec — what the documentation says

> The USE4 model contains **60 industry factors built on GICS**. The structure is a
> hybrid of GICS levels: some factors correspond to a single GICS industry or
> sub-industry, others aggregate or split GICS categories based on **economic
> intuition, statistical significance, explanatory power, and minimum size**.
> Thin industries are avoided because they produce unstable estimates.

> **Exposures:** most stocks have one primary industry exposure of 1.0. For
> diversified firms, USE4 estimates fractional exposures from business-segment
> reporting: industry-specific price-to-assets and price-to-sales coefficients
> from cross-sectional regressions, blended **75% assets / 25% sales**, top five
> segments retained and renormalized. ~63% of the estimation universe by cap
> weight had multiple exposures.

> **Identification:** every stock has country exposure 1 and industry exposures
> summing to 1, so the cross-sectional regression imposes a **cap-weighted
> zero-sum constraint on industry factor returns** — industry returns are pure
> performance relative to the market.

---

**What this spec builds is the exposure layer only.** The constraint and the
factor returns belong to the cross-sectional-regression stage (next repo);
the deliverable here must merely be regression-ready (complete coverage,
exposures summing to 1 — trivially true under single membership).

## §2. End-to-end pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│  UPSTREAM:  estu_build.ipynb  →  estu_monthly.parquet              │
├────────────────────────────────────────────────────────────────────┤
│  STAGE 0:  Setup, parameters, floor rule                           │
│  STAGE 1:  Inventory atoms → design + save your scheme CSV         │
│  STAGE 2:  Load shared ESTU                                        │
│  STAGE 3:  Classify: permaticker → atom → factor (your scheme)     │
│  STAGE 4:  Panel construction (static map × monthly calendar)      │
│  STAGE 5:  Save deliverable                                        │
│  STAGE 6:  Validate (8-check battery)                              │
└────────────────────────────────────────────────────────────────────┘
```

Stage 1 is the only stage where you make judgment calls rather than implement a spec. How many factors, which atoms to merge, which to split, what to name them, how to handle edge-case atoms. The output is a scheme CSV you own and version-control in this directory. Stages 3–6 are mechanical given the scheme.

Industries are **dummies, not descriptors**: no trim, no standardisation, no MoM ρ. The validation battery is membership-shaped (coverage, floor, churn, concentration) rather than distribution-shaped.

## §3. Output schema — what the build delivers

**File:** `data/out/industries_use4.parquet`

**Compression:** zstd, statistics=True

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date |
| `in_estu` | Bool | ESTU membership on signal_date |
| `mcap` | Float64 | Market cap on signal_date |
| `sharadar_industry` | String | The raw classification atom (audit column — keep, never drop) |
| `industry` | String | **The factor label** — one of your N scheme factors |
| `use4_sector` | String | Sector grouping of the factor (optional but useful for diagnostics) |

**Your scheme CSV** (save it adjacent to this notebook, version-control it): three required columns — `sharadar_industry` (raw atom), `factor` (the label to assign), `use4_sector` (broad sector grouping). One row per atom; every observed atom must appear — an unmapped atom is a hard failure. Add a boolean `floor_exception` column to mark factors kept despite failing the member floor.

**Coverage rules:**
- Every row of `estu_monthly` gets a label (coverage universe, not just ESTU).
- Single membership: exactly one row per (permaticker, signal_date).
- Exposures sum to 1 per stock by construction (trivially true under single membership).

### Second artifact: `industry_weights_use4.parquet`

| Column | Type | Description |
|---|---|---|
| `signal_date` | Date | End-of-month rebalance date |
| `industry` | String | Factor label |
| `n_members` | UInt32 | In-ESTU members on the date |
| `cap_weight` | Float64 | Industry share of total in-ESTU cap; sums to 1 per date (asserted < 1e-12) |

This is the $w$-vector for USE4's identification constraint $\sum_j w_j f_j^{\text{ind}} = 0$ — materialised here so the CSR (and the country-factor spec) never re-derive it ad hoc.

## §4. STAGE 0 — Setup, parameters, floor rule

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
SCHEME_PATH    = Path(__file__ or ".") / "your_scheme.csv"   # adjacent to notebook
CALENDAR_START = date(1999, 1, 1)

# 💡 DEFAULT (NOT IN PDF) — minimum-size floor for a viable factor
FLOOR_MEDIAN = 15   # median ESTU members over the full sample
FLOOR_MIN    = 8    # never fewer at any single date
```

❓ **NOT IN PDF — the floor values.** USE4 says only "thin industries are avoided because they produce unstable estimates." The values above are a reasonable starting point for ~2,500 ESTU members across a factor count near 55. Adjust if your factor count diverges significantly.

💡 **Exceptions to the floor:** some factors are economically important or explicitly called out in USE4 despite thin membership — cap weight is the relevant size for a cap-weighted regression, not headcount. Document any exceptions explicitly in your scheme CSV (the `floor_exception` column) and report their stats in the validation output.

## §5. STAGE 1 — Inventory atoms → design your scheme

This is the design stage. Read it before writing any code.

### What Sharadar gives you

✅ **PDF SPEC:** GICS is the classification foundation.
❌ **NOT IN DATA:** GICS is licensed and absent from Sharadar.

💡 **DEFAULT:** Sharadar's `industry` field in TICKERS. Load TICKERS and extract the distinct `industry` values — you will find roughly 150 atoms with Morningstar-style labels. These are the raw material.

Evaluate the alternatives before committing to `industry`:
- `famaindustry` (Fama–French 48): too coarse for a 50+ factor model — broad categories like "Business Services" hold hundreds of ESTU members.
- `sicsector` / `sicindustry`: SIC granularity reflects the 1980s economy; technology distinctions in particular are 40 years out of date.

Sharadar's `industry` field is the most granular and economically current atom set available without a licensed taxonomy.

🧪 **VALIDATE:** before designing the scheme, print the full atom inventory — distinct values, ESTU frequency, and member-count distribution. You will build the scheme from this list.

---

### How many factors?

USE4 uses 60. Your target is **somewhat less than 60** — the reasons are structural, not arbitrary:

1. **GICS-to-Sharadar mapping gaps.** Some USE4 categories cleanly map to Sharadar atoms; others do not. Where the atom granularity doesn't support a clean split, merging is the correct call (the USE4 criteria say so explicitly: economic intuition and explanatory power, not taxonomy fidelity).

2. **The 2011 vs 2026 universe.** USE4's scheme was engineered for the 2011 market. Several categories have hollowed out since — sectors that were viable in 2011 may fall below the member floor on today's ESTU. These must be merged rather than kept as ghost factors.

3. **Conversely**, some 2011 single-factor categories have become large enough to warrant splitting by modern standards (a 100+ member factor conflating two distinct business models is harder to interpret and estimate than two 50-member factors). These splits are legitimate and should be justified.

A reasonable target is **50–58 factors**, driven by applying USE4's own criteria to your atom inventory rather than hitting a specific number. Do not start from 60 and work backwards — start from the atom inventory and work up.

---

### Designing the scheme — what to decide for each atom

For every atom (or group of related atoms), ask:

- **Does it map cleanly to a USE4 60-list category?** If yes, and the member count is above the floor, use that category.
- **Is the atom too thin to stand alone?** Apply the floor. If it fails, which USE4 category is the closest economic neighbour? Merge there.
- **Does a single USE4 category have multiple Sharadar atoms with distinct enough return dynamics to split?** Check member counts on both halves. If both clear the floor, splitting is the better model.
- **Is the atom genuinely ambiguous?** A few atoms span multiple GICS categories by design (conglomerates, diversified financials). Make a documented judgment call; flag it in the scheme CSV.

---

### The fail-loudly constraint (non-negotiable)

Your build must fail hard on any atom present in TICKERS that is not in your scheme CSV. It must never silently default to `UNASSIGNED`. The reason: Sharadar occasionally updates its taxonomy. A new atom must be reviewed and assigned deliberately — silent bucketing produces an invisible, undocumented deviation.

The `UNASSIGNED` fallback in your scheme should exist as an explicit escape valve (for stocks with null `industry` in TICKERS), not as a catch-all for unmapped atoms. In-ESTU `UNASSIGNED` should be zero or near-zero; if it isn't, your TICKERS loading or dedup has a bug.

🧪 **VALIDATE:**
- All observed atoms covered by your scheme — unmapped atom ⇒ loud failure
- Your N factors + 1 fallback label present; every factor has ≥ 1 atom
- TICKERS dedup: one row per permaticker (current-snapshot; latest row wins on any duplicates)

## §6. STAGE 2 — Load the shared ESTU (`estu_monthly.parquet`)

✅ **PDF SPEC:** exposures are estimated for the estimation universe.

💡 **DEFAULT:** the shared `data/out/estu_monthly.parquet` from
`estu_build.ipynb` — same universe as every style factor, which the joint
cross-sectional regression requires.

🧪 **VALIDATE:** 330 monthly dates (1999-01-29 →), ESTU size mean ≈ 2,500.

## §7. STAGES 3–4 — Classification and panel

✅ **PDF SPEC:** one primary industry exposure of 1.0 per stock; fractional for diversified firms via segment data (75% assets / 25% sales).

❌ **NOT IN DATA:** SF1 carries no business-segment reporting — the fractional machinery is unbuildable. **Single 0/1 membership for every stock.** This is a headline deviation (USE4 reports ~63% of cap weight multi-exposed); the bias concentrates in conglomerates, diversified financials, and integrated energy.

❓ **NOT IN PDF — classification timing.** TICKERS is a current snapshot, not point-in-time: today's label applied to 1999 introduces mild look-ahead for reclassified/pivoted companies.
💡 **DEFAULT:** static classification, documented; **revisit trigger**: a PIT layer from EDGAR 10-K header SIC codes (free, filing-dated) if classification drift is shown to matter.

**Mechanics:** join each permaticker to your scheme CSV via `industry` atom → factor label. Null atom → `UNASSIGNED` (should be near zero in-ESTU; assert ≤ 2). The map is static per the snapshot TICKERS, so the panel is the scheme crossed with the monthly calendar.

🧪 **VALIDATE:**
- 100% of rows labelled; in-ESTU `UNASSIGNED` ≤ 2 per date
- Exactly one row per (permaticker, signal_date)
- Per-permaticker label is constant over time (static map ⇒ any label variation is a join/dedup bug, not a reclassification)

## §8. STAGES 5–6 — Save and validate

**Save:** zstd parquet, 7-column dtype assertion, read-back assertion.

**The 8-check battery:**

```
1   Coverage          in-ESTU UNASSIGNED rows ≤ 2 at every date
2   Completeness      all N factors have ≥ 1 ESTU member at every date
3   Floor             non-exception factors: median ≥ 15 and min ≥ 8 members;
                      declared exceptions reported with their stats
4   Static map        every permaticker carries exactly one factor label
5   Single membership exactly one row per (permaticker, signal_date)
6   Concentration     largest factor ≤ 30% of ESTU cap at every date
7   Spot checks       a handful of large, unambiguous names whose industry
                      assignment should be obvious (pick your own)
8   Disk ≡ memory     read-back equivalence
```

For check 7: choose 5–8 mega-cap names whose sector is unambiguous from their business description alone. If any spot check fails, the join or the scheme has a mistake — do not adjust the scheme to pass the spot check without understanding why.

## §9. Master list of ❓ NOT-IN-PDF / ❌ NOT-IN-DATA decisions

| # | Decision | Default chosen | Revisit when |
|---|---|---|---|
| 1 | GICS unavailable | Sharadar `industry` atoms + your hand-engineered scheme targeting the USE4 60-list | GICS or a crosswalk is licensed |
| 2 | No segment data | Single 0/1 membership for all stocks (USE4: ~63% of cap multi-exposed) | Segment-level data procured |
| 3 | Classification not PIT | Static current-snapshot labels | EDGAR 10-K SIC PIT layer if drift shown to matter |
| 4 | Floor rule | median ≥ 15 and min ≥ 8 ESTU members over full sample | CSR-based explanatory-power screening replaces member counts |
| 5 | Floor exceptions | Declared in your scheme CSV; cap weight is the relevant size | Any falls below ~5 members persistently |
| 6 | Merges vs the 60-list | Your call — apply USE4's criteria: economic coherence, atom granularity, member floor | CSR explanatory power favors a split AND members support it |
| 7 | Splits vs the 60-list | Your call — apply USE4's criteria to 2026 universe tectonics, not 2011 | Never on fidelity grounds — splits address real economic distinctions |
| 8 | Judgment atoms | Document your choices in the scheme CSV | Reclassification evidence |

## §10. Common pitfalls — read this before coding

**Pitfall 1: silently bucketing unmapped atoms.** A Sharadar taxonomy update can introduce new `industry` values. The build must FAIL on an unmapped atom, not default it — your scheme CSV must be reviewed and updated deliberately. This is the guard that makes the scheme maintainable.

**Pitfall 2: classification churn masquerading as universe churn.** With a static map, a permaticker's factor can never change. If membership counts jump without ESTU entry/exit, the join is broken (duplicate TICKERS rows are the usual cause — dedup per permaticker first).

**Pitfall 3: treating the fallback as a factor.** `UNASSIGNED` rows are kept in the deliverable (audit trail) but must be excluded from the CSR exposure matrix.

**Pitfall 4: forgetting the constraint downstream.** Country exposure ≡ 1 plus single industry membership ⇒ perfect collinearity. The CSR must impose the cap-weighted zero-sum constraint on industry returns; nothing in THIS deliverable prevents the mistake.

**Pitfall 5: judging factors by member count alone.** Some factors may have few members but enormous cap weight. Cap weight is the relevant size for a cap-weighted regression; the member floor guards estimate stability, not economic importance.

## §11. Final summary — what "done" looks like

1. ✅ Your scheme CSV: all observed atoms mapped, N factors documented, exceptions flagged
2. ✅ `industries_use4.parquet` matches the §3 schema
3. ✅ All 8 validation checks pass
4. ✅ All N factors present every date; floor exceptions reported with member stats
5. ✅ Spot checks land (names you selected; assignment matches your scheme)

Downstream: the CSR pivots `industry` to one-hot, joins the style exposures, adds the country column of ones, and imposes the cap-weighted zero-sum industry constraint using `industry_weights_use4.parquet`.